# Piste 8 — Assistant IA pour les techniciens (RAG, étape de recherche)

**Question métier :** un technicien pose une question en langage courant ; le système retrouve les passages du manuel qui y répondent, **avec la page**.

Le RAG (*Retrieval-Augmented Generation*) se fait en deux temps : (1) **retrouver** les passages pertinents, (2) faire **rédiger** la réponse par un LLM à partir de ces passages seulement, ce qui limite les hallucinations et rend la réponse vérifiable. Ce notebook réalise l'étape 1.

> ✅ **Document réel** : *California Commercial Driver Handbook* (DMV de Californie, 208 pages), dont la section 5 porte sur les **freins à air**, le même système que l'étude de cas Scania APS. Le PDF n'est pas inclus dans le dépôt : télécharge-le sur le site du DMV californien et place-le dans `data/comlhdbk.pdf`.

In [1]:
import re
import numpy as np
import pandas as pd
from pypdf import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ---------- 1. Lire le manuel et le découper en passages ----------
lecteur = PdfReader("data/comlhdbk.pdf")
passages = []
for num_page, page in enumerate(lecteur.pages, start=1):
    texte = re.sub(r"\s+", " ", page.extract_text() or "").strip()
    mots = texte.split()
    # passages d'environ 120 mots, qui se chevauchent de 40 mots pour ne pas couper une idée
    for debut in range(0, max(len(mots), 1), 80):
        morceau = " ".join(mots[debut:debut + 120])
        if len(morceau.split()) >= 30:
            passages.append({"page": num_page, "texte": morceau})
passages = pd.DataFrame(passages)
print(f"{len(lecteur.pages)} pages découpées en {len(passages)} passages")

# ---------- 2. Le moteur de recherche (étape « Retrieval » du RAG) ----------
vectoriseur = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), sublinear_tf=True)
matrice = vectoriseur.fit_transform(passages["texte"])

def rechercher(question, k=3):
    scores = cosine_similarity(vectoriseur.transform([question]), matrice).ravel()
    meilleurs = scores.argsort()[::-1][:k]
    return passages.iloc[meilleurs].assign(score=scores[meilleurs].round(3))

def assistant(question, k=3):
    print("=" * 100)
    print("QUESTION :", question)
    for _, r in rechercher(question, k).iterrows():
        print(f"\n  → page {r['page']} du PDF (pertinence {r['score']})")
        print("   ", r["texte"][:400], "…")

# ---------- 3. Questions d'un technicien sur les freins à air ----------
for q in [
    "How do I check the air brake system for leaks?",
    "What does the low air pressure warning signal mean?",
    "How often should I drain the air tanks and why?",
    "When do the spring brakes come on automatically?",
]:
    assistant(q)


208 pages découpées en 1326 passages
QUESTION : How do I check the air brake system for leaks?

  → page 137 du PDF (pertinence 0.241)
    the trailers. 7.3.2 – Additional Things to Check During a Walk Around Inspection Do these checks in addition to Section 5, Inspecting Air Brake Systems. 7.4 – Doubles/Triples Air Brake Check Check the brakes on a double or triple trailer as you would any combination vehicle. Section 6 explains how to check air brakes on combination vehicles. You must also make the following checks on your double o …

  → page 123 du PDF (pertinence 0.241)
    the brakes are released. — Check the air brake system pressure gauge for signs of major air loss. • When you are sure the trailer brakes are work - ing, start the engine. • Make sure the air pressure is up to normal. 6-9 …

  → page 186 du PDF (pertinence 0.203)
    the pedal at this position. • Check that the warning buzzer or light is off. Note: Failure to perform both components of the hy- draulic brake chec

## Résultats et limite
- **3 bonnes réponses sur 4**, page à l'appui : seuil d'alerte de basse pression (55 à 75 psi, p. 103), vidange quotidienne des réservoirs (p. 120 et 102), serrage automatique des freins à ressort (20 à 45 psi, p. 104).
- **Échec instructif** sur la question des fuites : la réponse est page 108 (test de fuite, perte maximale de 3 psi par minute pour un véhicule seul), mais le manuel écrit *leakage* là où la question dit *leaks*. Le moteur TF-IDF cherche des **mots**, pas des **idées**.

## Amélioration identifiée : la recherche sémantique
Des *embeddings* (par exemple le modèle `all-MiniLM-L6-v2` de `sentence-transformers`) représentent le **sens** de chaque passage, si bien que *leaks* et *leakage* se rapprochent. Code prévu (non exécuté ici : PyTorch n'est pas disponible pour Python 3.13 sur Mac Intel) :

```python
from sentence_transformers import SentenceTransformer
modele = SentenceTransformer("all-MiniLM-L6-v2")
vecteurs = modele.encode(passages["texte"].tolist(), normalize_embeddings=True)

def rechercher_sens(question, k=3):
    q = modele.encode([question], normalize_embeddings=True)[0]
    scores = vecteurs @ q
    meilleurs = scores.argsort()[::-1][:k]
    return passages.iloc[meilleurs].assign(score=scores[meilleurs].round(3))
```

Étape suivante : envoyer les passages retrouvés à un LLM (via une API) avec la consigne de répondre uniquement à partir d'eux, en citant les pages.